# Electricity Price Forecasting

## Project Overview

Forecasts **daily electricity prices** ($/MWh) over a 14-day horizon. Synthetic data spans 2 years with weekly patterns, seasonal variation, and price spikes.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily electricity prices, predict the next 14 days. Price forecasting enables optimal generation scheduling, storage dispatch, and trading strategies.

## Why This Project Matters

Electricity prices are highly volatile. Accurate forecasts enable generators to maximize revenue, retailers to manage risk, and consumers to optimize usage. Even small improvements translate to millions in savings.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily average electricity prices
- Weekly pattern (higher weekdays)
- Seasonal variation (summer/winter peaks)
- Random price spikes (weather events, supply disruptions)
- No negative prices (simplified)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `price_mwh` | Daily average electricity price ($/MWh) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'price_mwh'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 45 + 0.005 * t
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 5
    else: weekly[i] = -8

seasonal = 10 * np.sin(2 * np.pi * (t - 30) / 365)

# Price spikes
spikes = np.zeros(N_POINTS)
for s in range(0, N_POINTS, 120):
    spike_day = min(s + np.random.randint(0, 60), N_POINTS - 2)
    for j in range(min(2, N_POINTS - spike_day)):
        spikes[spike_day + j] = 40 + np.random.uniform(0, 30)

noise = np.random.normal(0, 5, N_POINTS)
price_mwh = base + weekly + seasonal + spikes + noise
price_mwh = np.maximum(price_mwh, 10).round(2)

df = pd.DataFrame({'date': dates, 'price_mwh': price_mwh})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,price_mwh
0,2022-01-01,19.87
1,2022-01-02,35.23
2,2022-01-03,44.12
3,2022-01-04,44.71
4,2022-01-05,38.31


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('price_mwh Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("price_mwh Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

price_mwh Statistics:
count    730.000000
mean      49.292658
std       13.021641
min       12.710000
25%       41.367500
50%       48.240000
75%       56.710000
max      127.160000
Name: price_mwh, dtype: float64

CV: 0.264
Skewness: 1.301


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       20.3 | RMSE:       28.1 | MAPE: 32.91%
Seasonal Naive (7)             | MAE:       12.4 | RMSE:       24.0 | MAPE: 16.88%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                           Adjusted R-Squared   R-Squared        RMSE  Time Taken
Model                                                                            
Lars                              3657.696822 -280.284371  108.168722    0.018449
LarsCV                            2303.601446 -176.123188   85.835434    0.034366
KernelRidge                        863.170795  -65.320830   52.523526    0.047407
GaussianProcessRegressor            63.657320   -3.819794   14.159344    0.073244
DummyRegressor                      29.271707   -1.174747    9.511154    0.013077
QuantileRegressor                   26.068909   -0.928378    8.956224    0.072198
ExtraTreeRegressor                  21.942352   -0.610950    8.185966    0.018100
LGBMRegressor                       17.299871   -0.253836    7.221864    0.096697
GradientBoostingRegressor           17.033144   -0.233319    7.162532    0.378428
DecisionTreeRegressor               15.723877   -0.132606    6.863858    0.02

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: extra_tree
FLAML (extra_tree)             | MAE:        4.6 | RMSE:        5.2 | MAPE: 10.62%
FLAML Test (extra_tree)        | MAE:       12.0 | RMSE:       23.7 | MAPE: 16.06%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       11.8 | RMSE:       22.5 | MAPE: 15.86%
SF AutoETS                     | MAE:       12.4 | RMSE:       25.0 | MAPE: 16.08%
SF SeasonalNaive               | MAE:       12.5 | RMSE:       25.3 | MAPE: 16.51%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                  model       MAE      RMSE      MAPE
     FLAML (extra_tree)  4.565638  5.222408 10.616722
           SF AutoARIMA 11.771807 22.479336 15.864014
FLAML Test (extra_tree) 11.992652 23.660776 16.058588
             SF AutoETS 12.352203 24.960797 16.084982
     Seasonal Naive (7) 12.411429 23.950074 16.881778
       SF SeasonalNaive 12.451429 25.290883 16.508668
     Naive (Last Value) 20.348571 28.087603 32.905761

Top 3:
                  model       MAE      RMSE      MAPE
     FLAML (extra_tree)  4.565638  5.222408 10.616722
           SF AutoARIMA 11.771807 22.479336 15.864014
FLAML Test (extra_tree) 11.992652 23.660776 16.058588


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 9.60, Std: 21.63


## Interpretation and Business Insight

- Electricity prices have weekday/weekend and seasonal patterns
- Price spikes are the hardest part to forecast
- Summer and winter peaks reflect demand-driven pricing
- Lag features capture mean-reversion dynamics
- Statistical models handle the seasonal component well

## Limitations

1. Synthetic — real prices depend on fuel costs, renewables, transmission
2. No negative prices (common in high-renewables markets)
3. No hourly granularity (prices vary dramatically intraday)
4. No fuel price or generation mix features

## How to Improve This Project

1. Add fuel price (gas, coal) features
2. Use hourly data for day-ahead market bidding
3. Model price spikes separately with extreme value distributions
4. Add renewable generation forecasts (solar, wind)
5. Use probabilistic forecasting for risk management

## Production Considerations

- Day-ahead price forecasting for market participation
- Integration with trading platforms
- Spike probability alerting
- Portfolio optimization using price scenarios

## Common Mistakes

1. Using point forecasts without uncertainty for trading decisions
2. Not handling price spikes separately
3. Ignoring fuel price dynamics
4. Using daily averages when hourly prices matter

## Mini Challenge / Exercises

1. Detect and separately model price spike events
2. Add a synthetic fuel price feature
3. Try quantile regression for price interval forecasts
4. Build a simple trading strategy using forecasts

## Final Summary / Key Takeaways

1. Electricity price forecasting is high-value for energy markets
2. Weekly and seasonal patterns provide a baseline
3. Price spikes are the key challenge — mean-reverting but hard to predict
4. Probabilistic forecasts are essential for trading
5. Real deployment needs hourly granularity and fuel price inputs

In [18]:
import json
metrics = {
    'project': 'Electricity Price Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Electricity Price Forecasting — Complete ===")

Metrics saved.

=== Electricity Price Forecasting — Complete ===
